# SQL for Bioinformatics

## Learning Objectives
1. Write SQL queries to retrieve biological data
2. Use JOINs to combine related tables
3. Aggregate and filter data with GROUP BY and HAVING
4. Integrate SQL with Python using sqlite3 and pandas

## Why SQL in Bioinformatics?

Many of the most important biological databases are relational databases at their core. Ensembl stores genome annotations across hundreds of species in MySQL. The UCSC Genome Browser exposes its data through a public MySQL server. PDB (Protein Data Bank) organizes structural data in tables linked by accession IDs. Even when you access these databases through a web interface or REST API, SQL is the language running underneath.

Understanding SQL gives you two practical advantages:

1. **Direct database access** — you can query Ensembl or UCSC directly with SQL, bypassing rate-limited APIs and downloading exactly the columns you need.
2. **Better pandas intuition** — pandas `merge`, `groupby`, and `agg` are a direct translation of SQL `JOIN`, `GROUP BY`, and aggregate functions. Knowing SQL makes pandas click.

In this notebook we use Python's built-in `sqlite3` module with an in-memory database — no server installation required.

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Create a small genomics database
cursor.executescript('''
CREATE TABLE genes (
    gene_id INTEGER PRIMARY KEY,
    symbol TEXT NOT NULL,
    chromosome TEXT,
    start_pos INTEGER,
    end_pos INTEGER,
    strand TEXT CHECK(strand IN ('+', '-')),
    biotype TEXT
);

CREATE TABLE expression (
    expr_id INTEGER PRIMARY KEY,
    gene_id INTEGER REFERENCES genes(gene_id),
    sample_id TEXT,
    tissue TEXT,
    tpm REAL,
    condition TEXT
);

CREATE TABLE variants (
    variant_id INTEGER PRIMARY KEY,
    gene_id INTEGER REFERENCES genes(gene_id),
    position INTEGER,
    ref_allele TEXT,
    alt_allele TEXT,
    clinical_significance TEXT
);
''')

# Insert sample data
genes_data = [
    (1, 'BRCA1', 'chr17', 43044295, 43125483, '-', 'protein_coding'),
    (2, 'TP53',  'chr17', 7668402,  7687550,  '-', 'protein_coding'),
    (3, 'EGFR',  'chr7',  55019017, 55211628, '+', 'protein_coding'),
    (4, 'KRAS',  'chr12', 25204789, 25250936, '-', 'protein_coding'),
    (5, 'MYC',   'chr8',  127735434, 127742951, '+', 'protein_coding'),
    (6, 'PTEN',  'chr10', 87863113, 87971930, '+', 'protein_coding'),
    (7, 'RB1',   'chr13', 48303747, 48481890, '-', 'protein_coding'),
    (8, 'BRAF',  'chr7',  140719327, 140924929, '-', 'protein_coding'),
    (9, 'MALAT1','chr11', 65497688, 65506516, '+', 'lncRNA'),
    (10,'MIR21', 'chr17', 59841266, 59841337, '+', 'miRNA'),
]
cursor.executemany('INSERT INTO genes VALUES (?,?,?,?,?,?,?)', genes_data)

# Expression data across tissues and conditions
import random
random.seed(42)
expr_id = 1
for gene_id in range(1, 11):
    for tissue in ['breast', 'lung', 'colon', 'brain']:
        for condition in ['normal', 'tumor']:
            base_tpm = random.uniform(1, 100)
            if condition == 'tumor' and gene_id in [1, 2, 5]:  # DE genes
                base_tpm *= random.uniform(2, 5)
            cursor.execute('INSERT INTO expression VALUES (?,?,?,?,?,?)',
                           (expr_id, gene_id, f'S{expr_id:03d}', tissue, round(base_tpm, 2), condition))
            expr_id += 1

# Variants
variants_data = [
    (1,  1, 43091434, 'A', 'G', 'pathogenic'),
    (2,  1, 43094464, 'C', 'T', 'pathogenic'),
    (3,  2, 7674220,  'G', 'A', 'pathogenic'),
    (4,  2, 7675088,  'C', 'T', 'likely_pathogenic'),
    (5,  3, 55191822, 'T', 'G', 'drug_response'),
    (6,  4, 25227342, 'G', 'T', 'pathogenic'),
    (7,  4, 25227341, 'G', 'A', 'pathogenic'),
    (8,  6, 87933147, 'C', 'T', 'pathogenic'),
    (9,  8, 140753336,'A', 'T', 'pathogenic'),
    (10, 2, 7676381,  'G', 'C', 'benign'),
]
cursor.executemany('INSERT INTO variants VALUES (?,?,?,?,?,?)', variants_data)
conn.commit()
print('Database created with 10 genes, 80 expression records, 10 variants')

## Basic Queries: SELECT, WHERE, ORDER BY

The core of SQL is `SELECT ... FROM ... WHERE`. Use `ORDER BY` to sort results and `LIMIT` to cap the number of rows returned.

In [ ]:
# All genes on chromosome 17
df = pd.read_sql_query(
    "SELECT symbol, chromosome, start_pos, end_pos FROM genes WHERE chromosome = 'chr17'",
    conn
)
print(df)

# Genes longer than 100 kb, ordered by length
df = pd.read_sql_query("""
    SELECT symbol, chromosome, (end_pos - start_pos) AS length
    FROM genes
    WHERE (end_pos - start_pos) > 100000
    ORDER BY length DESC
""", conn)
print(df)

## Aggregate Functions and GROUP BY

`COUNT`, `AVG`, `MAX`, `MIN`, and `SUM` summarize groups of rows. `GROUP BY` splits the table into groups before aggregation. `HAVING` filters groups after aggregation (equivalent to `WHERE` on the aggregated result).

In [ ]:
# Average expression per tissue and condition
df = pd.read_sql_query("""
    SELECT tissue, condition, ROUND(AVG(tpm), 2) AS avg_tpm, COUNT(*) AS n_samples
    FROM expression
    GROUP BY tissue, condition
    ORDER BY tissue, condition
""", conn)
print(df)

# Genes with highest average tumor expression (HAVING filters after grouping)
df = pd.read_sql_query("""
    SELECT g.symbol, ROUND(AVG(e.tpm), 2) AS avg_tumor_tpm
    FROM genes g
    JOIN expression e ON g.gene_id = e.gene_id
    WHERE e.condition = 'tumor'
    GROUP BY g.symbol
    HAVING AVG(e.tpm) > 50
    ORDER BY avg_tumor_tpm DESC
""", conn)
print(df)

## JOIN Operations

JOINs combine rows from two tables based on a related column.

- **INNER JOIN** — only rows that have a match in both tables.
- **LEFT JOIN** — all rows from the left table; NULLs where no match exists in the right table.
- **Self-join** — a table joined to itself, useful for comparing rows within the same table.

In [ ]:
# INNER JOIN: pathogenic variants with gene names
df = pd.read_sql_query("""
    SELECT g.symbol, g.chromosome, v.position, v.ref_allele, v.alt_allele, v.clinical_significance
    FROM variants v
    INNER JOIN genes g ON v.gene_id = g.gene_id
    WHERE v.clinical_significance = 'pathogenic'
    ORDER BY g.symbol
""", conn)
print(df)

# LEFT JOIN: all genes with variant count (including genes with 0 variants)
df = pd.read_sql_query("""
    SELECT g.symbol, COUNT(v.variant_id) AS n_variants
    FROM genes g
    LEFT JOIN variants v ON g.gene_id = v.gene_id
    GROUP BY g.symbol
    ORDER BY n_variants DESC
""", conn)
print(df)

## Subqueries

A subquery is a `SELECT` nested inside another query. Use them with `IN` to filter rows against a dynamically computed set — handy when you want to combine criteria from different tables without an explicit JOIN.

In [ ]:
# Genes that are both highly expressed in tumors AND carry pathogenic variants
df = pd.read_sql_query("""
    SELECT symbol FROM genes
    WHERE gene_id IN (
        SELECT gene_id FROM expression
        WHERE condition = 'tumor'
        GROUP BY gene_id
        HAVING AVG(tpm) > 50
    )
    AND gene_id IN (
        SELECT gene_id FROM variants
        WHERE clinical_significance = 'pathogenic'
    )
""", conn)
print('Highly expressed tumor genes with pathogenic variants:')
print(df)

## CREATE, INSERT, UPDATE, DELETE

SQL is not read-only. You can create new tables to store analysis results, insert rows, update values, and delete records. This is useful for storing intermediate results from a pipeline directly in a database.

In [ ]:
# Add a new analysis results table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS de_results (
        gene_id INTEGER REFERENCES genes(gene_id),
        log2fc REAL,
        pvalue REAL,
        padj REAL,
        significant INTEGER
    )
''')

# Insert differential expression results
de_data = [
    (1, 2.3,  0.001,  0.01,  1),
    (2, 1.8,  0.005,  0.03,  1),
    (3, 0.2,  0.45,   0.78,  0),
    (5, 3.1,  0.0001, 0.002, 1),
]
cursor.executemany('INSERT INTO de_results VALUES (?,?,?,?,?)', de_data)
conn.commit()

# Query the results joined back to gene names
df = pd.read_sql_query("""
    SELECT g.symbol, d.log2fc, d.padj,
           CASE WHEN d.significant = 1 THEN 'Yes' ELSE 'No' END AS significant
    FROM de_results d
    JOIN genes g ON d.gene_id = g.gene_id
    ORDER BY d.padj
""", conn)
print(df)

## Exercises

**Exercise 1** (★) — Write a query that returns all `protein_coding` genes on chromosome 7, showing their symbol, start position, end position, and strand.

In [ ]:
# Your query here
df = pd.read_sql_query("""
    -- TODO
""", conn)
print(df)

**Exercise 2** (★★) — Calculate the fold change (tumor AVG TPM / normal AVG TPM) for each gene in breast tissue. Return gene symbol and fold change, ordered by fold change descending.

In [ ]:
# Your query here
df = pd.read_sql_query("""
    -- TODO
""", conn)
print(df)

**Exercise 3** (★★) — Find genes that have more than one pathogenic variant. Return gene symbol and the count of pathogenic variants, ordered by count descending.

In [ ]:
# Your query here
df = pd.read_sql_query("""
    -- TODO
""", conn)
print(df)

conn.close()

---

[← Previous: Module 18: Data Visualization](../../Tier_1_Python_for_Bioinformatics/18_Data_Visualization/01_data_visualization.ipynb) | [Back to Course Overview](../../README.md)

---